# Dataset Adaptation Summary

Le dataset public **Brazilian E-commerce (Olist)** a été utilisé comme source de données pour ce projet, les données réelles de Carrefour ne pouvant pas être utilisées pour des raisons de confidentialité.

Afin de l’adapter à un contexte de **grande distribution et de programme de fidélité**, plusieurs transformations ont été réalisées.

Le modèle a été simplifié en transformant les **orders en transactions** et en supprimant certaines tables non pertinentes pour l’analyse (**géolocalisation**, **avis clients**, **vendeurs**).

Un **système de fidélité simulé** a été introduit avec la création :
- d’un **foyer client** (`foyer_id`)
- d’une **carte de fidélité** (`card_id`)
- d’un **statut de carte** (`card_status`)

Les **statuts de transaction** ont été regroupés en trois catégories :

- **VALIDATED**
- **CANCELLED**
- **PENDING**

Les **moyens de paiement** ont été simplifiés en :
- **card**
- **cash**
- **other**

Une variable **channel** a été ajoutée pour simuler un environnement **omnicanal** :
- `store`
- `online`

Les dates du dataset initial ont été **décalées vers une période récente (2024–2026)** et une variable **quantity** a été simulée afin de représenter un comportement d’achat réaliste en grande distribution.

À l’issue de ces transformations, trois datasets principaux ont été générés :

- **transactions_enriched**
- **transaction_items**
- **products**


In [ ]:
import os
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

PROJECT_DIR = r"C:\Users\soufi\Documents\projet-carrefour"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
ARCHIVE_DIR = os.path.join(DATA_DIR, "archive")

IN_FILE = os.path.join(ARCHIVE_DIR, "olist_customers_dataset.csv")

Charger customers (Olist)

In [ ]:
df = pd.read_csv(IN_FILE)
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
df.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state'], dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [ ]:
# création du foyer
df["master_customer_id"] = df["customer_unique_id"]

customers_clean = df[[
    "customer_id",
    "master_customer_id",
    "customer_city",
    "customer_state"
]].copy()

customers_clean.head()

,customer_id,master_customer_id,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


In [ ]:
# Nombre total de foyers
customers_clean["master_customer_id"].nunique()

96096

In [ ]:
# Taille max d’un foyer
customers_clean.groupby("master_customer_id")["customer_id"].count().max()

np.int64(17)

In [ ]:
# Calcul taille de chaque foyer
foyer_sizes = customers_clean.groupby("master_customer_id")["customer_id"].count()

# Comptage des tailles
distribution = foyer_sizes.value_counts().sort_index()

distribution

customer_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

Charger orders (Olist)

In [ ]:
ORDERS_FILE = os.path.join(ARCHIVE_DIR, "olist_orders_dataset.csv")
orders = pd.read_csv(ORDERS_FILE)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

Convertir le timestamp

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max()

(Timestamp('2016-09-04 21:15:19'), Timestamp('2018-10-17 17:30:18'))

Créer transactions_clean 

In [ ]:
transactions_clean = orders.rename(columns={
    "order_id": "transaction_id",
    "order_status": "transaction_status",
    "order_purchase_timestamp": "transaction_timestamp"
})[["transaction_id", "customer_id", "transaction_status", "transaction_timestamp"]].copy()

transactions_clean.head()

,transaction_id,customer_id,transaction_status,transaction_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39


In [ ]:
transactions_clean["transaction_status"].value_counts()

transaction_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Mapper les statuts :

delivered, shipped, invoiced, approved → VALIDATED

canceled, unavailable → CANCELLED

processing, created → PENDING

In [ ]:
status_map = {
    "delivered": "VALIDATED",
    "shipped": "VALIDATED",
    "invoiced": "VALIDATED",
    "approved": "VALIDATED",
    "canceled": "CANCELLED",
    "unavailable": "CANCELLED",
    "processing": "PENDING",
    "created": "PENDING"
}

transactions_clean["transaction_status"] = (
    transactions_clean["transaction_status"]
        .map(status_map)
)

In [ ]:
transactions_clean["transaction_status"].value_counts()

transaction_status
VALIDATED    97901
CANCELLED     1234
PENDING        306
Name: count, dtype: int64

Joindre transactions avec customers

In [ ]:
transactions_enriched = transactions_clean.merge(
    customers_clean,
    on="customer_id",
    how="left"
)

In [ ]:
transactions_enriched.isna().sum()

transaction_id           0
customer_id              0
transaction_status       0
transaction_timestamp    0
master_customer_id       0
customer_city            0
customer_state           0
dtype: int64

In [ ]:
transactions_enriched.shape

(99441, 7)

Préparer payment_type (card/cash)

In [ ]:
PAYMENTS_FILE = os.path.join(ARCHIVE_DIR, "olist_order_payments_dataset.csv")
payments = pd.read_csv(PAYMENTS_FILE)
payments.head()


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
payments["payment_type"].unique()

array(['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined'],
      dtype=object)

In [ ]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Mapping pour le contexte grande distribution (Carrefour simulé) :

credit_card → carte bancaire → card

debit_card → carte bancaire → card

boleto → paiement type ticket bancaire → cash

voucher → bon d’achat → cash

not_defined → rare anomalie → other 

In [ ]:
def map_payment_type(pt):
    pt = str(pt).lower().strip()

    if pt in ["credit_card", "debit_card"]:
        return "card"
    elif pt in ["boleto", "voucher"]:
        return "cash"
    else:
        return "other"

payments["payment_type_simple"] = payments["payment_type"].apply(map_payment_type)

payments["payment_type_simple"].value_counts()

payment_type_simple
card     78324
cash     25559
other        3
Name: count, dtype: int64

Agrégation du type de payement par transaction

In [ ]:
payments_agg = (
    payments.groupby("order_id")["payment_type_simple"]
    .apply(lambda s: ",".join(sorted(set(s))))
    .reset_index()
    .rename(columns={
        "order_id": "transaction_id",
        "payment_type_simple": "payment_types_used"
    })
)

payments_agg.head()

,transaction_id,payment_types_used
0,00010242fe8c5a6d1ba2dd792cb16214,card
1,00018f77f2f0320c557190d7a144bdd3,card
2,000229ec398224ef6ca0657da4fc703e,card
3,00024acbcdf0a6daa1e931b038114c75,card
4,00042b26cf59d7ce69dfabb4e55b4fd9,card


In [ ]:
payments_agg.shape

(99440, 2)

In [ ]:
transactions_enriched = transactions_enriched.merge(
    payments_agg,
    on="transaction_id",
    how="left"
)

transactions_enriched["payment_types_used"] = (
    transactions_enriched["payment_types_used"]
    .fillna("other")
)

In [ ]:
transactions_enriched.columns

Index(['transaction_id', 'customer_id', 'transaction_status', 'transaction_timestamp', 'master_customer_id', 'customer_city', 'customer_state',
       'payment_types_used'],
      dtype='object')

Décalage temporel du Dataset

In [ ]:
import pandas as pd

max_date = transactions_enriched["transaction_timestamp"].max()
today = pd.Timestamp.today()

delta = today - max_date
delta

Timedelta('2693 days 16:55:48.739537')

In [ ]:
transactions_enriched["transaction_timestamp"] = (
    transactions_enriched["transaction_timestamp"] + delta
)

In [ ]:
transactions_enriched["transaction_timestamp"].min(), \
transactions_enriched["transaction_timestamp"].max()

(Timestamp('2024-01-20 14:11:07.739537'),
 Timestamp('2026-03-03 10:26:06.739537'))

*** Création logique de la date de création carte ***

La logique métier réaliste : La date de création carte = première transaction du client

In [ ]:
card_creation = (
    transactions_enriched.groupby("customer_id")["transaction_timestamp"]
    .min()
    .reset_index()
    .rename(columns={"transaction_timestamp": "issued_at"})
)

In [ ]:
import pandas as pd

max_date = transactions_enriched["transaction_timestamp"].max()
today = pd.Timestamp.today()

delta = today - max_date
delta

Timedelta('2693 days 16:33:21.880395')

Renommer master_customer_id → foyer_id

In [ ]:
transactions_enriched = transactions_enriched.rename(columns={"master_customer_id": "foyer_id"})

*** Générer la colonne "channel"***
Règles que je veux appliquer:

En se basant sur payment_types_used (agrégé par transaction) :

Si payment_types_used == "cash" → channel = "store"

Si payment_types_used == "card,cash" (ou cash+card) → channel = "store"

Si payment_types_used == "card" → 50% store, 50% online

(Optionnel) Si other ou valeurs rares → on met store (par défaut)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def assign_channel(payment_types_used):
    s = str(payment_types_used).lower().strip()

    # cas cash ou mix => store
    if s == "cash" or "cash" in s:
        return "store"

    # cas card uniquement => 50/50
    if s == "card":
        return "store" if rng.random() < 0.5 else "online"

    # cas autres / inconnus
    return "store"

transactions_enriched["channel"] = transactions_enriched["payment_types_used"].apply(assign_channel)

In [ ]:
transactions_enriched["channel"].value_counts(normalize=True) * 100

channel
store     61.685824
online    38.314176
Name: proportion, dtype: float64

L’identifiant de carte est généré de manière déterministe à partir du customer_id 
La logique: 
1 client = 1 carte valide

In [ ]:
transactions_enriched["card_id"] = transactions_enriched["customer_id"].apply(lambda x: f"CARD_{x}")

transactions_enriched[["customer_id", "card_id"]].head()

,customer_id,card_id
0,9ef432eb6251297304e76186b10a928d,CARD_9ef432eb6251297304e76186b10a928d
1,b0830fb4747a6c6d20dea0b8c802d7ef,CARD_b0830fb4747a6c6d20dea0b8c802d7ef
2,41ce2a54c0b03bf3443c3d931a367089,CARD_41ce2a54c0b03bf3443c3d931a367089
3,f88197465ea7920adcdbec7375364d82,CARD_f88197465ea7920adcdbec7375364d82
4,8ab97904e6daea8866dbdbc4fb7aad2c,CARD_8ab97904e6daea8866dbdbc4fb7aad2c


Création de la colonne issued_at (pour Loyalty_cards):
date d’émission de la carte = première transaction du client

In [ ]:
issued_at_map = (
    transactions_enriched
    .groupby("customer_id")["transaction_timestamp"]
    .min()
    .rename("issued_at")
)

In [ ]:
transactions_enriched["issued_at"] = transactions_enriched["customer_id"].map(issued_at_map)

In [ ]:
transactions_enriched[["customer_id", "transaction_timestamp", "issued_at"]].head(10)

,customer_id,transaction_timestamp,issued_at
0,9ef432eb6251297304e76186b10a928d,2025-02-16 03:52:21.739537,2025-02-16 03:52:21.739537
1,b0830fb4747a6c6d20dea0b8c802d7ef,2025-12-08 13:37:25.739537,2025-12-08 13:37:25.739537
2,41ce2a54c0b03bf3443c3d931a367089,2025-12-23 01:34:37.739537,2025-12-23 01:34:37.739537
3,f88197465ea7920adcdbec7375364d82,2025-04-04 12:23:54.739537,2025-04-04 12:23:54.739537
4,8ab97904e6daea8866dbdbc4fb7aad2c,2025-06-30 14:14:27.739537,2025-06-30 14:14:27.739537
5,503740e9ca751ccdda7ba28e9ab8f608,2024-11-23 14:52:53.739537,2024-11-23 14:52:53.739537
6,ed0271e0b7da060a393796590e7b737a,2024-08-26 05:17:56.739537,2024-08-26 05:17:56.739537
7,9bdf08b4b3b52b5526ff42d37d47f222,2024-09-30 06:06:18.739537,2024-09-30 06:06:18.739537
8,f54a9f0e6b351c431402b8461ea51999,2024-06-09 11:24:57.739537,2024-06-09 11:24:57.739537
9,31ad1d1b63eb9962463f764d4e6e0c9d,2024-12-13 04:50:50.739537,2024-12-13 04:50:50.739537


Creation de la colonne "card_status"
card_status = active 
pour toutes les cartes car le dataset source ne contient pas d’historique d’opposition / expiration / remplacement (on simule donc uniquement des cartes actives).

In [ ]:
transactions_enriched["card_status"] = "active"
transactions_enriched["card_status"].value_counts()

card_status
active    99441
Name: count, dtype: int64

Création de la colonne "foyer_created_at":
La date de création du foyer est 24h avant la première carte émise au sein du foyer.

In [ ]:
# Calculer la première carte du foyer

foyer_first_card_map = (
    transactions_enriched
    .groupby("foyer_id")["issued_at"]
    .min()
)

In [ ]:
# Ajouter foyer_created_at

transactions_enriched["foyer_created_at"] = (
    transactions_enriched["foyer_id"]
    .map(foyer_first_card_map)
    - pd.Timedelta(hours=24)
)

In [ ]:
transactions_enriched[[
    "foyer_id",
    "issued_at",
    "foyer_created_at"
]].head(10)

,foyer_id,issued_at,foyer_created_at
0,7c396fd4830fd04220f754e42b4e5bff,2025-02-16 03:52:21.739537,2025-01-18 04:22:26.739537
1,af07308b275d755c9edb36a90c618231,2025-12-08 13:37:25.739537,2025-12-07 13:37:25.739537
2,3a653a41f6f9fc3d2a113cf8398680e8,2025-12-23 01:34:37.739537,2025-12-22 01:34:37.739537
3,7c142cf63193a1473d2e66489a9ae977,2025-04-04 12:23:54.739537,2025-04-03 12:23:54.739537
4,72632f0f9dd73dfee390c9b22eb56dd6,2025-06-30 14:14:27.739537,2025-06-29 14:14:27.739537
5,80bb27c7c16e8f973207a5086ab329e2,2024-11-23 14:52:53.739537,2024-11-22 14:52:53.739537
6,36edbb3fb164b1f16485364b6fb04c73,2024-08-26 05:17:56.739537,2024-08-25 05:17:56.739537
7,932afa1e708222e5821dac9cd5db4cae,2024-09-30 06:06:18.739537,2024-09-29 06:06:18.739537
8,39382392765b6dc74812866ee5ee92a7,2024-06-09 11:24:57.739537,2024-06-08 11:24:57.739537
9,299905e3934e9e181bfb2e164dd4b4f8,2024-12-13 04:50:50.739537,2024-12-12 04:50:50.739537


Chargement du fichier "olist_order_items_dataset.csv"

In [ ]:
ITEMS_FILE = os.path.join(ARCHIVE_DIR, "olist_order_items_dataset.csv")
items = pd.read_csv(ITEMS_FILE)

items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
# Renommer les colonnes
items_clean = items.rename(columns={
    "order_id": "transaction_id",
    "order_item_id": "transaction_item_id",
    "price": "unit_price"
})[["transaction_id", "transaction_item_id", "product_id", "unit_price"]].copy()

items_clean.head()

,transaction_id,transaction_item_id,product_id,unit_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,239.90
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,199.90


Logique réaliste de la quantité de produit acheté :

1 dans la majorité des cas

parfois 2–3

rarement 4–10

In [ ]:
rng = np.random.default_rng(42)

def simulate_quantity(n):
    values = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    probs  = np.array([0.85, 0.10, 0.03, 0.015, 0.004, 0.0002, 0.0002, 0.0002, 0.0002, 0.0002])
    probs = probs / probs.sum()   # sécurité : somme = 1
    return rng.choice(values, size=n, p=probs)

items_clean["quantity"] = simulate_quantity(len(items_clean))

# Vérifier la répartition
(items_clean["quantity"].value_counts(normalize=True).sort_index() * 100).round(3)

quantity
1     84.991
2      9.958
3      3.053
4      1.507
5      0.389
6      0.026
7      0.017
8      0.020
9      0.020
10     0.020
Name: proportion, dtype: float64

Lire le fichier products

In [ ]:
PRODUCTS_FILE = os.path.join(ARCHIVE_DIR, "olist_products_dataset.csv")
products = pd.read_csv(PRODUCTS_FILE)

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [ ]:
products.columns
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [ ]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")

In [ ]:
products = products[[
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].copy()
products.head()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0


In [ ]:
products["product_category_name"].value_counts()

product_category_name
cama_mesa_banho                  3029
esporte_lazer                    2867
moveis_decoracao                 2657
beleza_saude                     2444
utilidades_domesticas            2335
                                 ... 
fashion_roupa_infanto_juvenil       5
casa_conforto_2                     5
pc_gamer                            3
seguros_e_servicos                  2
cds_dvds_musicais                   1
Name: count, Length: 74, dtype: int64

Exporter transactions

In [ ]:
transactions_enriched.columns

Index(['transaction_id', 'customer_id', 'transaction_status', 'transaction_timestamp', 'foyer_id', 'customer_city', 'customer_state', 'payment_types_used',
       'channel', 'card_id', 'issued_at', 'card_status', 'foyer_created_at'],
      dtype='object')

In [ ]:
transactions_enriched.to_csv(os.path.join(DATA_DIR, "transactions_enriched.csv"), index=False)

Exporter transaction_items

In [ ]:
items_clean.columns

Index(['transaction_id', 'transaction_item_id', 'product_id', 'unit_price', 'quantity'], dtype='object')

In [ ]:
items_clean.to_csv(os.path.join(DATA_DIR, "transaction_items_clean.csv"), index=False)

Exporter products

In [ ]:
products.columns

Index(['product_id', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'], dtype='object')

In [ ]:
products.to_csv(os.path.join(DATA_DIR, "products_clean.csv"), index=False)